# HyperGrid Model Labs 🚀
---
Klasifikasi Kondisi Permukaan Jalan & Deteksi Lubang (*Pothole Detection*) berbasis Sensor Inersial Smartphone (Akselerometer & Giroskop).

### Pipeline Arsitektur:
1. **Penyelarasan Ground-Truth:** Pemetaan timestamp lubang per perjalanan (*trip-based*).
2. **Invariansi Orientasi:** Ekstraksi magnitudo 3D $\sqrt{x^2+y^2+z^2}$ & Dynamic Acceleration.
3. **Sliding Time Window:** Ekstraksi 32 fitur statistik dan sinyal (*Mean, Std, RMS, Jerk, ZCR, Crest Factor*).
4. **Group-Aware Splitting:** Pengujian pada perjalanan yang benar-benar belum pernah dilihat (*Unseen Trips*).
5. **Ekspor ONNX:** Model mandiri (*standalone*) untuk komputasi tepi (*Edge ML* di Android/iOS).

In [ ]:
# 1. Import Dependencies
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster, HeatMap
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve

from utils.dataset import load_raw_trips
from utils.features import RoadFeatureExtractor
from utils.trainer import train_hypergrid_model

print('All libraries and custom modules imported successfully.')

In [ ]:
# 2. Memuat Data Perjalanan & Penyelarasan Label Ground-Truth
trips = load_raw_trips(data_dir='./data', pothole_tolerance_sec=0.75)
print(f'Total Sesi Perjalanan yang Dimuat: {len(trips)} trips')

In [ ]:
# 3. Ekstraksi Fitur Sinyal Invarian Orientasi (Sliding Window)
extractor = RoadFeatureExtractor(window_size=20, step_size=5)
X, y, groups, metadata = extractor.extract_from_trips(trips)

print('=== HASIL EKSTRAKSI FITUR ===')
print(f'Matriks Fitur X : {X.shape}')
print(f'Label Target y  : {y.shape}')
print('\nDistribusi Label:')
print(y.value_counts())
print('\nContoh 5 Baris Pertama Fitur:')
X.head()

In [ ]:
# 4. Training Model XGBoost dengan Group-Aware Cross-Validation & Evaluasi Unseen Trips
model_path = './models/hg_model_v1.onnx'
results = train_hypergrid_model(
    X=X,
    y=y,
    groups=groups,
    metadata=metadata,
    model_output_path=model_path,
    test_size=0.2,
    random_state=42,
    tune_hyperparameters=True,
)

best_model = results['model']
optimal_threshold = results['optimal_threshold']
results_df = results['results_df']

In [ ]:
# 5. Visualisasi Feature Importance (Gain)
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette='viridis')
plt.title('Top 15 Fitur Paling Berpengaruh dalam Deteksi Pothole (Gain Score)', fontsize=13)
plt.xlabel('Importance Score')
plt.ylabel('Nama Fitur')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Visualisasi Evaluasi Model (Confusion Matrix & PR-Curve)
y_test_true = results_df['actual_label']
y_test_pred = results_df['predicted_label']
y_test_prob = results_df['pothole_probability']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test_true, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['Normal', 'Pothole'], yticklabels=['Normal', 'Pothole'])
axes[0].set_title(f'Confusion Matrix (Threshold: {optimal_threshold:.3f})')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test_true, y_test_prob)
axes[1].plot(rec, prec, color='#009EFF', lw=2, label='PR Curve')
axes[1].set_title('Precision-Recall Curve (Unseen Test Trips)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 7. Visualisasi Peta Spasial GIS (Folium Interactive Map)
df_map = results_df.dropna(subset=['latitude', 'longitude']).copy()
center_lat = df_map['latitude'].mean()
center_lon = df_map['longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='OpenStreetMap')
cluster = MarkerCluster(name='Titik Deteksi').add_to(m)

for idx, row in df_map.iterrows():
    is_pothole = row['predicted_label'] == 1
    color = '#e74c3c' if is_pothole else '#3498db'
    status = '⚠️ POTHOLE' if is_pothole else '✅ NORMAL'
    
    popup_content = f"<b>{status}</b><hr><b>Trip ID:</b> {row['trip_id']}<br><b>Probabilitas:</b> {row['pothole_probability']:.2%}<br><b>Peak Shock:</b> {row['peak_shock']:.2f} m/s²<br><b>Kecepatan:</b> {row['speed']:.1f} km/h"
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6 if is_pothole else 3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8 if is_pothole else 0.4,
        popup=folium.Popup(popup_content, max_width=250),
    ).add_to(cluster)

# Heatmap Sebaran Pothole
pothole_coords = df_map[df_map['predicted_label'] == 1][['latitude', 'longitude', 'pothole_probability']].values.tolist()
if len(pothole_coords) > 0:
    HeatMap(pothole_coords, radius=15, blur=10).add_to(m)

folium.LayerControl().add_to(m)
print('Peta spasial interaktif berhasil dibuat!')
m